# Assignment 3 — Colab Setup, Data Staging & Checkpoint Sync

Shared setup notebook for **Member A & Member B** (CO3133 Semester 261 — Group RTX5090).

What this notebook does:
1. **Mount Google Drive** and clone/pull the repository into `/content/`.
2. **Install / verify pinned dependencies** from `assignment3/requirements.txt` and print **GPU / CUDA / PyTorch** and library versions.
3. **Copy data archives from Drive to Colab local disk BEFORE extracting** (reading thousands of small PNGs directly from Drive is slow).
4. **Checkpoint & results sync plan**: pull prior `last.pt` / `best.pt` from Drive before resuming, and sync `checkpoints/<run_id>/` + `results/<run_id>/` back to Drive after each run or epoch.

### Recorded Reference Hardware & Pinned Environment
| Environment | GPU | Driver / CUDA | Python | PyTorch / Torchvision |
| :--- | :--- | :--- | :--- | :--- |
| Local (Member B) | NVIDIA GeForce RTX 5070 Ti (16 GB) | Driver 616.56 / CUDA 13.0 (`>= 12.8` required) | 3.12.9 | `2.11.0+cu130` / `0.26.0+cu130` |
| Google Colab | L4 / A100 / T4 / RTX PRO 6000 (session-dependent) | Printed in Section 2 | Colab Python 3.x | Verified in Section 2 |

> **Tip for Member A:** If the dataset folder `KITTI_DC` lives on Member B's Drive, open the shared folder in Google Drive and choose **Organize → Add shortcut to Drive** under `MyDrive/KITTI_DC` so the exact same paths work without changes.

In [ ]:
# @title 1. Configuration & Mount Google Drive
REPO_URL = "https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/CO3133_sem261_groupRTX5090"  # @param {type:"string"}

# Google Drive paths
DRIVE_KITTI_DIR = "/content/drive/MyDrive/KITTI_DC"  # @param {type:"string"}
DRIVE_RUNS_DIR = "/content/drive/MyDrive/KITTI_DC/a3_runs"  # @param {type:"string"}

# Local fast-disk paths on Colab
LOCAL_DATA_DIR = "/content/kitti"  # @param {type:"string"}
LOCAL_TMP_DIR = "/content/tmp_archives"  # @param {type:"string"}

# Dataset archive source: 'dataset_tars' uses train.tar/val.tar/test.tar; 'raw_zips' copies raw zips locally first
DATA_SOURCE = "dataset_tars"  # @param ["dataset_tars", "raw_zips"]

# Set True to install the exact pinned stack (torch==2.11.0+cu130 + numpy==2.2.6); False keeps Colab's matched preinstalled stack
FORCE_REINSTALL_TORCH = False  # @param {type:"boolean"}

from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_RUNS_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_RUNS_DIR}/results", exist_ok=True)
print("Drive mounted. Runs backup dir:", DRIVE_RUNS_DIR)

In [ ]:
# @title 2. Clone or Pull Repository
import os, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

A3_DIR = f"{REPO_DIR}/assignment3"
%cd {A3_DIR}
!git rev-parse --short HEAD

In [ ]:
# @title 3. Install Dependencies & Record GPU / CUDA / PyTorch Info
import importlib.metadata as meta
import os, platform, subprocess, sys

req_path = f"{A3_DIR}/requirements.txt"
with open(req_path) as f:
    req_lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]

if FORCE_REINSTALL_TORCH:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_path], check=True)
else:
    # Keep Colab's preinstalled torch + numpy/scipy stack to avoid C-ABI mismatches; install only missing packages
    installed = {d.metadata["Name"].lower() for d in meta.distributions() if d.metadata["Name"]}
    missing = [
        ln for ln in req_lines
        if not ln.startswith("--extra-index-url")
        and ln.split("==")[0].strip().lower() not in installed
    ]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np, torch, torchvision
assert torch.from_numpy(np.zeros((2, 3), dtype=np.float32)).numpy().shape == (2, 3), "NumPy/PyTorch ABI mismatch!"
print("=== Hardware & Runtime ===")
print("Python      :", platform.python_version())
print("CPU count   :", os.cpu_count())
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU         : {props.name} ({props.total_memory / 1024**3:.2f} GB VRAM)")
    # Quick CUDA + AMP smoke test at KITTI resolution (352 x 1216)
    x = torch.randn(1, 3, 352, 1216, device="cuda")
    with torch.amp.autocast("cuda"):
        y = torch.nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3).cuda()(x)
    print("CUDA smoke  :", tuple(y.shape), "OK")

print("\n=== Installed Library Versions ===")
for pkg in ["torch", "torchvision", "numpy", "pillow", "pyyaml", "tqdm", "matplotlib", "scipy", "pandas", "opencv-python", "pytest"]:
    try:
        print(f"{pkg:15s} == {meta.version(pkg)}")
    except meta.PackageNotFoundError:
        print(f"{pkg:15s} : NOT INSTALLED")

## 4. Stage Data to Colab Local Disk Before Extracting

- **Why copy first:** extracting directly from `/content/drive/...` reads/writes through the Drive FUSE mount, which is very slow and prone to timeouts.
- **Default (`dataset_tars`):** copies `KITTI_DC/dataset/{train,val,test}.tar` (~10 GB total) from Drive to `/content/tmp_archives/`, extracts them into `/content/kitti/`, and removes the temporary `.tar` copies.
- **`raw_zips` mode:** copies the 3 depth zips (`data_depth_annotated.zip`, `data_depth_velodyne.zip`, `data_depth_selection.zip`) from `KITTI_DC/zips/` to local disk before extracting (raw drive zips can be staged drive-by-drive with `scripts/prepare_data.py`).

In [ ]:
# @title 4. Copy Archives to Local Disk & Extract
import os, shutil, subprocess, time

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Colab local disk free: {free_gb:.1f} GB")
os.makedirs(LOCAL_TMP_DIR, exist_ok=True)

if DATA_SOURCE == "dataset_tars":
    tar_dir = f"{DRIVE_KITTI_DIR}/dataset"
    if all(os.path.isdir(f"{LOCAL_DATA_DIR}/{s}/rgb") for s in ("train", "val", "test")):
        print(f"Dataset already present at {LOCAL_DATA_DIR}; skipping copy/extract.")
    else:
        assert free_gb >= 22, f"Need >= 22 GB free on /content, only {free_gb:.1f} GB available"
        os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
        for split in ("train", "val", "test"):
            src_tar = f"{tar_dir}/{split}.tar"
            local_tar = f"{LOCAL_TMP_DIR}/{split}.tar"
            assert os.path.exists(src_tar), f"Missing {src_tar}. Run src/kitti_prepare_dataset_colab.ipynb first."
            t0 = time.time()
            print(f"Copying {split}.tar to local disk...")
            shutil.copy2(src_tar, local_tar)
            print(f"Extracting {split}.tar to {LOCAL_DATA_DIR}...")
            subprocess.run(["tar", "-xf", local_tar, "-C", LOCAL_DATA_DIR], check=True)
            os.remove(local_tar)
            print(f"Done {split} in {time.time() - t0:.1f}s")

    for split in ("train", "val", "test"):
        n_rgb = len(os.listdir(f"{LOCAL_DATA_DIR}/{split}/rgb"))
        n_lid = len(os.listdir(f"{LOCAL_DATA_DIR}/{split}/lidar"))
        n_gt  = len(os.listdir(f"{LOCAL_DATA_DIR}/{split}/gt"))
        print(f"{split:5s}: rgb={n_rgb}, lidar={n_lid}, gt={n_gt}")

    # Symlink assignment3/data/processed -> LOCAL_DATA_DIR so configs/base.yaml works without --data-dir
    os.makedirs(f"{A3_DIR}/data", exist_ok=True)
    repo_data_link = f"{A3_DIR}/data/processed"
    if os.path.islink(repo_data_link):
        if os.readlink(repo_data_link) != LOCAL_DATA_DIR:
            os.unlink(repo_data_link)
            os.symlink(LOCAL_DATA_DIR, repo_data_link)
    elif not os.path.exists(repo_data_link):
        os.symlink(LOCAL_DATA_DIR, repo_data_link)
    print(f"Data path linked: {repo_data_link} -> {os.path.realpath(repo_data_link)}")

elif DATA_SOURCE == "raw_zips":
    zip_dir = f"{DRIVE_KITTI_DIR}/zips"
    raw_out = f"{LOCAL_DATA_DIR}_raw"
    os.makedirs(raw_out, exist_ok=True)
    for zname in ("data_depth_selection.zip", "data_depth_velodyne.zip", "data_depth_annotated.zip"):
        src_zip = f"{zip_dir}/{zname}"
        local_zip = f"{LOCAL_TMP_DIR}/{zname}"
        assert os.path.exists(src_zip), f"Missing {src_zip}"
        if not os.path.exists(local_zip):
            print(f"Copying {zname} to local disk ({os.path.getsize(src_zip) / 1e9:.2f} GB)...")
            shutil.copy2(src_zip, local_zip)
        print(f"Staged locally at {local_zip} (extract drive-by-drive with scripts/prepare_data.py)")

## 5. Checkpoint & Results Persistence Plan (Google Drive)

Colab runtimes reset when idle or when GPU quota expires, so **all runs must be backed up to Google Drive**:
1. **Before resuming a run:** call `restore_run(run_id)` to copy `checkpoints/<run_id>/` (`last.pt`, `best.pt`, `log.csv`, `config.yaml`) and `results/<run_id>/` from `DRIVE_RUNS_DIR` to local `/content/`.
2. **During / after training:** run `backup_run(run_id)` (or pass Drive backup / symlink `checkpoints` -> `DRIVE_RUNS_DIR/checkpoints` if desired) so `best.pt`, `last.pt`, `log.csv`, `metrics_test.json`, `per_range.csv`, and `results/summary.csv` are persisted.
3. **Before closing the tab:** run `drive.flush_and_unmount()` or verify files appear in Drive.

> **Symlink option vs. copy sync:** Local writes + periodic `backup_run(run_id)` avoid Drive I/O stalls during training. If you want automatic per-epoch persistence without modifying `train.py`, run `link_outputs_to_drive()` below before launching training (writes go straight to Drive, then call `drive.flush_and_unmount()` at the end).

In [ ]:
# @title 5. Checkpoint Sync & Symlink Helpers
import os, shutil

def restore_run(run_id: str):
    """Copy saved checkpoints and results for <run_id> from Drive to local repo."""
    for kind in ("checkpoints", "results"):
        src = f"{DRIVE_RUNS_DIR}/{kind}/{run_id}"
        dst = f"{A3_DIR}/{kind}/{run_id}"
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Restored {src} -> {dst}")
        else:
            print(f"No remote {kind} found for {run_id} (starting fresh)")

def backup_run(run_id: str):
    """Sync local checkpoints and results for <run_id> (plus summary.csv) to Drive."""
    for kind in ("checkpoints", "results"):
        src = f"{A3_DIR}/{kind}/{run_id}"
        dst = f"{DRIVE_RUNS_DIR}/{kind}/{run_id}"
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Backed up {src} -> {dst}")
    summary_src = f"{A3_DIR}/results/summary.csv"
    if os.path.exists(summary_src):
        shutil.copy2(summary_src, f"{DRIVE_RUNS_DIR}/results/summary.csv")
        print("Backed up results/summary.csv")

def link_outputs_to_drive(enable: bool = False):
    """Optional: symlink assignment3/checkpoints and results directly to Drive."""
    if not enable:
        print("Symlink skipped (using manual restore_run / backup_run).")
        return
    for kind in ("checkpoints", "results"):
        local_path = f"{A3_DIR}/{kind}"
        drive_path = f"{DRIVE_RUNS_DIR}/{kind}"
        if os.path.islink(local_path):
            print(f"{local_path} already linked -> {os.readlink(local_path)}")
            continue
        if os.path.isdir(local_path):
            shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
            shutil.rmtree(local_path)
        os.symlink(drive_path, local_path)
        print(f"Linked {local_path} -> {drive_path}")

# Example usage:
# link_outputs_to_drive(enable=False)
# restore_run("m2_lidar_352x1216_s42")
# !python -m src.train --config configs/base.yaml
# backup_run("m2_lidar_352x1216_s42")
print("Sync helpers ready. Example: restore_run('<run_id>'), backup_run('<run_id>')")